# Model Optimization: Quantization

In this notebook, we'll apply quantization techniques to our models using distributed processing. Instead of running the quantization on our notebook instance, we'll launch separate SageMaker Processing jobs to perform the quantization on more powerful instances.

## What is Quantization?

Quantization is a technique that reduces the precision of the numbers used to represent a model's parameters. For example, converting 32-bit floating point numbers to 8-bit integers. This significantly reduces model size and can improve inference speed, often with minimal impact on accuracy.

### Benefits of Quantization:
- **Reduced Model Size**: Smaller models require less storage and memory
- **Faster Inference**: Lower precision calculations can be faster, especially on hardware with specialized support
- **Lower Memory Bandwidth**: Smaller models require less memory bandwidth, which can be a bottleneck
- **Energy Efficiency**: Lower precision calculations consume less power

### Types of Quantization We'll Explore:
- **Dynamic Quantization**: Weights are quantized ahead of time, but activations are quantized dynamically during inference
- **Static Quantization**: Both weights and activations are quantized ahead of time
- **Quantization-Aware Training**: The model is trained with simulated quantization to improve accuracy

### Distributed Processing Approach
This notebook uses SageMaker Processing jobs to perform quantization on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
import osimport jsonimport torchimport timeimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport boto3import sagemakerfrom sagemaker.processing import ProcessingInput, ProcessingOutput, Processorfrom sagemaker.pytorch.processing import PyTorchProcessor# Import our utility functionsfrom optimization_utils import analyze_job_failure, handle_processing_error# Import our utility functionsfrom optimization_utils import analyze_job_failure, save_checkpoint, handle_processing_error

# Import our utility functions


## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    
    # Set default values that user should update
    S3_BUCKET = "YOUR_BUCKET_NAME_HERE"  # Update this value
    AWS_REGION = "YOUR_REGION_HERE"      # Update this value
    SAGEMAKER_ROLE_ARN = "YOUR_ROLE_ARN_HERE"  # Update this value
    OPTIMIZATION_INSTANCE_TYPE = "ml.c5.xlarge"  # Default optimization instance type
    
    # Store the updated values
    %store S3_BUCKET
    %store AWS_REGION
    %store SAGEMAKER_ROLE_ARN
    %store OPTIMIZATION_INSTANCE_TYPE


## 3. Load Baseline Metrics and Model Information

In [ ]:
# Load baseline metrics from file
with open('baseline_metrics.json', 'r') as f:
    baseline_metrics = json.load(f)

print(f"Loaded baseline metrics for {len(baseline_metrics)} models")

# Load model information from file
with open('model_info.json', 'r') as f:
    model_info = json.load(f)

print(f"Loaded information for {len(model_info)} models")


## 4. Define Sample Inputs for Each Task

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment_analysis": "I really enjoyed this movie. The acting was superb and the plot was engaging.",
    "ner": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington.",
    "question_answering": {
        "question": "What is machine learning?",
        "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
    },
    "masked_lm": "The [MASK] is a large language model trained by OpenAI."
}


## 5. Create and Upload Quantization Script to S3

In this section, we'll examine and upload the Python script that performs the actual quantization. This script will be executed on the SageMaker Processing instances.

### What the Script Does:
1. **Loads the model and tokenizer** from Hugging Face
2. **Prepares sample inputs** for inference
3. **Applies quantization** using the specified method and bit precision
4. **Measures performance metrics** like model size and inference time
5. **Saves the quantized model** and metrics to the output directory

The script supports different quantization methods:
- **Dynamic Quantization**: Converts weights to INT8 at runtime (fastest to apply, good for RNNs and transformers)
- **Static Quantization**: Pre-computes quantization for both weights and activations (requires calibration data)
- **Quantization-Aware Training**: Simulates quantization during training to improve accuracy (requires retraining)

In [ ]:
# Display the quantization script with syntax highlighting
%pycat quantization_script.py


In [ ]:
# Upload the quantization script to S3
s3_client = boto3.client('s3')
s3_client.upload_file(
    'quantization_script.py', 
    S3_BUCKET, 
    'scripts/quantization_script.py'
)

print(f"Uploaded quantization script to s3://{S3_BUCKET}/scripts/quantization_script.py")


## 6. Launch Distributed Quantization Jobs

Now we'll set up and launch the SageMaker Processing jobs to perform quantization. Each model will be processed in a separate job, allowing for parallel processing.

### SageMaker Processing Components:
- **PyTorchProcessor**: A specialized processor for PyTorch workloads
- **Instance Type**: We'll use the instance type specified in the workshop settings
- **Role**: The SageMaker execution role with necessary permissions
- **Base Job Name**: A prefix for the SageMaker job names

The processor will run our quantization script on the specified instance type, with access to the model information stored in S3.

In [ ]:
# Define the instance type to use for quantization
instance_type = OPTIMIZATION_INSTANCE_TYPE
print(f"Using instance type: {instance_type} for optimization jobs")

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="1.13.1",
    py_version="py39",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="model-quantization",
    sagemaker_session=sagemaker_session,
    # Add required packages
    dependencies=["transformers", "datasets"]
)


In [ ]:
# Launch quantization jobs for each modelquantization_jobs = {}for model_key in model_info.keys():    print(f"\nLaunching quantization job for {model_key}...")        # Save model info to a temporary file    with open(f'temp_{model_key}_info.json', 'w') as f:        json.dump({model_key: model_info[model_key]}, f)        # Upload to S3    s3_client.upload_file(        f'temp_{model_key}_info.json',         S3_BUCKET,         f'optimization/inputs/{model_key}/model_info.json'    )        # Define inputs and outputs    inputs = [        ProcessingInput(            source=f's3://{S3_BUCKET}/scripts/quantization_script.py',            destination='/opt/ml/processing/input/code/quantization_script.py'        ),        ProcessingInput(            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',            destination='/opt/ml/processing/input/data/model_info.json'        )    ]        outputs = [        ProcessingOutput(            source='/opt/ml/processing/output',            destination=f's3://{S3_BUCKET}/optimization/outputs/{model_key}'        )    ]        # Run the processing job    job = processor.run(        code='/opt/ml/processing/input/code/quantization_script.py',        inputs=inputs,        outputs=outputs,        arguments=[            '--model-info-path', '/opt/ml/processing/input/data/model_info.json',            '--output-dir', '/opt/ml/processing/output',            '--quantization-method', 'dynamic',            '--quantization-bits', '8'        ]    )        # Store the job    quantization_jobs[model_key] = job    print(f"Launched quantization job: {job.job_name}")        # Clean up temporary file    os.remove(f'temp_{model_key}_info.json')    try:            ],            logs=True  # Enable CloudWatch Logs streaming


## 7. Monitor Job Status

After launching the quantization jobs, we need to monitor their progress. SageMaker Processing jobs run asynchronously, so we'll periodically check their status until all jobs are complete.

### Job Status Lifecycle:
- **InProgress**: The job is currently running
- **Completed**: The job has successfully completed
- **Failed**: The job encountered an error and failed
- **Stopping**: The job is in the process of stopping
- **Stopped**: The job was manually stopped

We'll display a table showing the status of each job and update it every 30 seconds until all jobs are complete.

In [ ]:
# Monitor job statusimport time# Create a SageMaker clientsagemaker_client = boto3.client('sagemaker')# Check job status every 30 secondsall_completed = Falsewhile not all_completed:    all_completed = True    job_statuses = {}        for model_key, job in quantization_jobs.items():        response = sagemaker_client.describe_processing_job(            ProcessingJobName=job.job_name        )        status = response['ProcessingJobStatus']        job_statuses[model_key] = status                if status in ['InProgress', 'Stopping']:            all_completed = False        # Display status table    status_df = pd.DataFrame({        'Model': list(job_statuses.keys()),        'Status': list(job_statuses.values())    })    display(status_df)        if not all_completed:        print("Waiting for jobs to complete...")        time.sleep(30)    else:        print("All jobs completed!")


## 8. Collect Results

Once all jobs are complete, we'll collect and combine the results from each job. Each job produces a metrics file containing information about the quantized model, such as size, inference time, and quantization parameters.

### Process:
1. Download the metrics file for each model from S3
2. Combine the metrics into a single dictionary
3. Save the combined metrics to a local file for use in later notebooks

This gives us a comprehensive view of the quantization results across all models.

In [ ]:
# Download and combine results
quantized_metrics = {}

for model_key in model_info.keys():
    # Download metrics file
    try:
        s3_client.download_file(
            S3_BUCKET,
            f'optimization/outputs/{model_key}/quantized_metrics.json',
            f'temp_{model_key}_quantized_metrics.json'
        )
        
        # Load metrics
        with open(f'temp_{model_key}_quantized_metrics.json', 'r') as f:
            metrics = json.load(f)
        
        # Add to combined metrics
        quantized_metrics.update(metrics)
        
        print(f"Downloaded metrics for {model_key}")
    except Exception as e:
        print(f"Error downloading metrics for {model_key}: {e}")

    # Save combined metrics
    with open('quantized_metrics.json', 'w') as f:
    json.dump(quantized_metrics, f, indent=2)

    print(f"\nSaved quantized metrics for {len(quantized_metrics)} models to quantized_metrics.json")


## 9. Compare Results

Now we'll compare the performance of the quantized models against the baseline models. This comparison helps us understand the impact of quantization on model size and inference speed.

### Key Metrics to Compare:
- **Model Size**: How much smaller are the quantized models?
- **Inference Time**: How much faster are the quantized models?
- **Size Reduction Percentage**: The percentage reduction in model size
- **Inference Speedup Percentage**: The percentage improvement in inference speed

We'll create a DataFrame to display these comparisons in a clear, tabular format.

In [ ]:
# Create a DataFrame for comparison
comparison_data = []

for model_key in quantized_metrics.keys():
    if model_key in baseline_metrics:
        baseline = baseline_metrics[model_key]
        quantized = quantized_metrics[model_key]
        
        # Calculate improvements
        size_reduction = (baseline['model_size'] - quantized['model_size']) / baseline['model_size'] * 100
        time_reduction = (baseline['inference_time'] - quantized['inference_time']) / baseline['inference_time'] * 100
        
        comparison_data.append({
            'Model': quantized['model_name'],
            'Quantization Method': quantized['quantization_method'],
            'Bits': quantized['quantization_bits'],
            'Baseline Size (MB)': baseline['model_size'],
            'Quantized Size (MB)': quantized['model_size'],
            'Size Reduction (%)': size_reduction,
            'Baseline Inference (ms)': baseline['inference_time'],
            'Quantized Inference (ms)': quantized['inference_time'],
            'Inference Speedup (%)': time_reduction
        })

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display the DataFrame
comparison_df


## 10. Deploy Models to SageMaker for Inference

Now that we've quantized our models, let's deploy them to SageMaker endpoints for real-world inference testing. We'll deploy both the original and quantized models to compare their performance.

### Deployment Process:
1. **Create model artifacts** in S3 for both original and quantized models
2. **Create SageMaker models** using these artifacts
3. **Create endpoints** to host the models
4. **Wait for endpoint deployment** to complete

This will allow us to directly compare inference performance between the original and quantized models.

In [ ]:
# Create a SageMaker client
sagemaker_client = boto3.client('sagemaker')

# Select a model to deploy for testing
model_key = list(quantized_metrics.keys())[0]  # Just use the first model for testing
print(f"Deploying model: {model_key} - {quantized_metrics[model_key]['model_name']}")

# Create model artifacts in S3
original_model_data = f"s3://{S3_BUCKET}/models/{model_key}/original"
quantized_model_data = f"s3://{S3_BUCKET}/models/{model_key}/quantized"

# Create SageMaker model for original model
original_model_name = f"{model_key}-original"
sagemaker_client.create_model(
    ModelName=original_model_name,
    PrimaryContainer={
        'Image': '763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-inference:1.13.1-transformers4.26.0-cpu-py39-ubuntu20.04',
        'ModelDataUrl': original_model_data,
        'Environment': {
            'HF_MODEL_ID': quantized_metrics[model_key]['model_name'],
            'HF_TASK': quantized_metrics[model_key]['task']
        }
    },
    ExecutionRoleArn=SAGEMAKER_ROLE_ARN
)

# Create SageMaker model for quantized model
quantized_model_name = f"{model_key}-quantized"
sagemaker_client.create_model(
    ModelName=quantized_model_name,
    PrimaryContainer={
        'Image': '763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-inference:1.13.1-transformers4.26.0-cpu-py39-ubuntu20.04',
        'ModelDataUrl': quantized_model_data,
        'Environment': {
            'HF_MODEL_ID': f"quantized-{quantized_metrics[model_key]['model_name']}",
            'HF_TASK': quantized_metrics[model_key]['task']
        }
    },
    ExecutionRoleArn=SAGEMAKER_ROLE_ARN
)

# Create endpoint configurations
original_endpoint_config_name = f"{model_key}-original-config"
sagemaker_client.create_endpoint_config(
    EndpointConfigName=original_endpoint_config_name,
    ProductionVariants=[
        {
            'VariantName': 'AllTraffic',
            'ModelName': original_model_name,
            'InstanceType': 'ml.m5.large',
            'InitialInstanceCount': 1
        }
    ]
)

quantized_endpoint_config_name = f"{model_key}-quantized-config"
sagemaker_client.create_endpoint_config(
    EndpointConfigName=quantized_endpoint_config_name,
    ProductionVariants=[
        {
            'VariantName': 'AllTraffic',
            'ModelName': quantized_model_name,
            'InstanceType': 'ml.m5.large',
            'InitialInstanceCount': 1
        }
    ]
)

# Create endpoints
original_endpoint_name = f"{model_key}-original-endpoint"
sagemaker_client.create_endpoint(
    EndpointName=original_endpoint_name,
    EndpointConfigName=original_endpoint_config_name
)

quantized_endpoint_name = f"{model_key}-quantized-endpoint"
sagemaker_client.create_endpoint(
    EndpointName=quantized_endpoint_name,
    EndpointConfigName=quantized_endpoint_config_name
)

print(f"Deploying original endpoint: {original_endpoint_name}")
print(f"Deploying quantized endpoint: {quantized_endpoint_name}")
print("Deployment in progress... This may take several minutes.")


In [ ]:
# Wait for endpoints to be in service
def wait_for_endpoint(endpoint_name):
    status = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)['EndpointStatus']
    print(f"Endpoint {endpoint_name} status: {status}")
    
    while status == 'Creating':
        time.sleep(30)
        status = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)['EndpointStatus']
        print(f"Endpoint {endpoint_name} status: {status}")
    
    return status

original_status = wait_for_endpoint(original_endpoint_name)
quantized_status = wait_for_endpoint(quantized_endpoint_name)

if original_status == 'InService' and quantized_status == 'InService':
    print("Both endpoints are ready for inference!")
else:
    print(f"Warning: Endpoints not in service. Original: {original_status}, Quantized: {quantized_status}")


## 11. Test Inference Performance

Now that our endpoints are deployed, let's test their inference performance. We'll send the same input to both endpoints and measure:
1. **Response time**: How long it takes to get a response
2. **Throughput**: How many requests can be processed per second
3. **Output quality**: Whether the outputs are similar between original and quantized models

This will give us a real-world comparison of the performance benefits of quantization.

In [ ]:
# Create a SageMaker runtime client for inference
runtime_client = boto3.client('sagemaker-runtime')

# Prepare input data based on the model task
task = quantized_metrics[model_key]['task']
if task == 'sequence-classification':
    input_data = {"inputs": sample_inputs["sentiment_analysis"]}
elif task == 'token-classification':
    input_data = {"inputs": sample_inputs["ner"]}
elif task == 'question-answering':
    input_data = {
        "inputs": {
            "question": sample_inputs["question_answering"]["question"],
            "context": sample_inputs["question_answering"]["context"]
        }
    }
elif task == 'masked-lm':
    input_data = {"inputs": sample_inputs["masked_lm"]}
else:
    input_data = {"inputs": "This is a test input."}

# Convert to JSON
payload = json.dumps(input_data)

# Function to measure inference time
def measure_endpoint_inference_time(endpoint_name, payload, num_runs=10):
    response_times = []
    responses = []
    
    for _ in range(num_runs):
        start_time = time.time()
        response = runtime_client.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Body=payload
        )
        end_time = time.time()
        
        response_time = (end_time - start_time) * 1000  # Convert to ms
        response_times.append(response_time)
        
        # Parse response
        response_body = response['Body'].read().decode('utf-8')
        responses.append(json.loads(response_body))
    
    avg_response_time = sum(response_times) / len(response_times)
    throughput = 1000 / avg_response_time  # Requests per second
    
    return {
        "avg_response_time": avg_response_time,
        "throughput": throughput,
        "responses": responses
    }

# Measure inference time for both endpoints
print("Testing original model endpoint...")
original_results = measure_endpoint_inference_time(original_endpoint_name, payload)

print("Testing quantized model endpoint...")
quantized_results = measure_endpoint_inference_time(quantized_endpoint_name, payload)

# Display results
print("\nInference Performance Results:")
print(f"Original Model - Avg Response Time: {original_results['avg_response_time']:.2f} ms, Throughput: {original_results['throughput']:.2f} req/s")
print(f"Quantized Model - Avg Response Time: {quantized_results['avg_response_time']:.2f} ms, Throughput: {quantized_results['throughput']:.2f} req/s")

# Calculate improvement
time_improvement = (original_results['avg_response_time'] - quantized_results['avg_response_time']) / original_results['avg_response_time'] * 100
throughput_improvement = (quantized_results['throughput'] - original_results['throughput']) / original_results['throughput'] * 100

print(f"\nImprovement with Quantization:")
print(f"Response Time: {time_improvement:.2f}% faster")
print(f"Throughput: {throughput_improvement:.2f}% higher")

# Compare outputs
print("\nSample Output Comparison:")
print(f"Original Model Output: {original_results['responses'][0]}")
print(f"Quantized Model Output: {quantized_results['responses'][0]}")


## 12. Clean Up Endpoints

To avoid unnecessary costs, let's clean up the endpoints we created. SageMaker endpoints incur charges as long as they're running, so it's important to delete them when they're no longer needed.

In [ ]:
# Delete endpoints and endpoint configurations
print("Cleaning up endpoints...")

# Delete endpoints
sagemaker_client.delete_endpoint(EndpointName=original_endpoint_name)
sagemaker_client.delete_endpoint(EndpointName=quantized_endpoint_name)
print(f"Deleted endpoints: {original_endpoint_name}, {quantized_endpoint_name}")

# Delete endpoint configurations
sagemaker_client.delete_endpoint_config(EndpointConfigName=original_endpoint_config_name)
sagemaker_client.delete_endpoint_config(EndpointConfigName=quantized_endpoint_config_name)
print(f"Deleted endpoint configurations: {original_endpoint_config_name}, {quantized_endpoint_config_name}")

# Delete models
sagemaker_client.delete_model(ModelName=original_model_name)
sagemaker_client.delete_model(ModelName=quantized_model_name)
print(f"Deleted models: {original_model_name}, {quantized_model_name}")

print("\nCleanup complete!")


## 13. Next Steps

Now that we've applied quantization to our models and tested their inference performance, we'll explore pruning techniques in the next notebook to further reduce model size.

### What We've Learned:
- How to apply quantization to transformer models
- How to use SageMaker Processing for distributed optimization tasks
- How quantization affects model size and inference speed
- How to deploy and test models on SageMaker endpoints
- The real-world performance benefits of quantization

### What's Next - Pruning:
Pruning is another optimization technique that removes unnecessary weights from the model. While quantization reduces the precision of weights, pruning removes weights entirely, making the model more sparse. The combination of quantization and pruning can lead to even greater size reductions and performance improvements.